# Research log: ERCOT day-ahead–real-time spreads and response surfaces

Aaron Munford, New York University

A dated record of what was done, what was decided and why, and what was learned, from the
start of the project to the final report (`reports/report_v2.pdf`). Decisions are tagged to
`DECISIONS.md` (D-xx). Every number quoted here comes from a CSV under `results/`; the
exploratory results are written up in `EDA_FINDINGS.md` and the model results in
`MODEL_RESULTS.md`. The companion notebook `02_analysis.ipynb` recomputes the report's tables
and figures.

**Contents**
1. 1 October: set-up, and the NCSS chapter
2. 9 October: the 2025 data
3. 10 October: exploratory analysis
4. 10 October: the response-surface ladder
5. 10 October: forecast surprises
6. 10 October: settled prices
7. 10 October: the 2024 replication and the West basis
8. 10 October: matched comparisons and the report's figures
9. 10 October: extreme hours and the 16 May event
10. 10 October: controls and ramps (isolated experiment)
11. 10 October: the final report
12. Open questions and next steps

## 1 October 2026: set-up

**The task.** Read the response surface regression chapter of the NCSS documentation (Chapter
330); collect hourly day-ahead (DA) and real-time (RT) prices and load for ERCOT's load zones
over a recent year; apply response surface regression with temperature, demand and natural gas
prices; and report the findings in at most ten pages.

**Built.**
* The repository skeleton, with every tunable choice in `config/study.toml`, each tagged to a
  decision in `DECISIONS.md`.
* The RSM engine (`rsm.py`), tested against quadratics with known stationary points,
  eigenvalues and ridge paths.
* The feature pipeline: the spread, implied heat rates, gas-to-power date alignment, load
  measures, zonal temperatures and daylight-saving handling.
* Fetchers for Grid Status, the Iowa Environmental Mesonet (ASOS), Open-Meteo and EIA.
* An end-to-end run on synthetic data, to test the plumbing before any real data existed.
  Synthetic outputs are kept out of `results/` and never used as evidence.

**Decisions.**
* **Study window: calendar 2025 (D-01).** A complete, settled year that sits almost entirely
  in one real-time market design.
* **RTC+B (D-02).** ERCOT's Real-Time Co-optimization Plus Batteries went live on operating day
  5 December 2025 (confirmed from ERCOT's news release of that day). It enters every regression
  as an intercept shift, a nuisance block rather than a finding (D-27).
* **Zones (D-03).** The four competitive load zones: Houston, North, South and West. Their
  names match ERCOT's load forecast zones, which is what makes a zonal join of price to load
  possible.

## 1 October 2026 (later): NCSS Chapter 330, and the engine checked against it

Read the chapter in full. Added the outputs the engine was missing: pure-error versions of the
ANOVA tables, the per-factor ANOVA, last R² per term, exact-duplicate lack of fit,
hierarchical elimination with an R² cutoff, the distinct-values guard and residual
diagnostics.

**Verification.** The engine reproduces NCSS's worked example (the Odor data) to every printed
digit: coefficients, standard errors, t and p values, last R², both ANOVA tables in both
versions, lack of fit (F = 40.75, p = 0.024047), the optimum at (0.1219, 0.1996, 1.7705)
with value −52.02463, and the prediction for row 16 (28.375). This is pinned as a regression
test (`tests/test_ncss_odor.py`).

### Reading notes: NCSS Chapter 330, *Response Surface Regression*

Page numbers are NCSS's own (330-n).

* **What it fits** (330-1): a polynomial with cross products, each factor raised to at most
  the third power, and hierarchical models only (every lower-order term that can be formed by
  reducing exponents is also in the model). The payoff is stability, freedom to recode a
  variable, and a clean relationship with ANOVA.
* **Framing** (330-1): a polynomial approximation of the true nonlinear model, like a Taylor
  expansion, which works well in a specified region; a smaller region needs fewer terms.
  *Here:* a quadratic over a whole year of Texas weather covers a large region, so
  out-of-sample checks matter.
* **Model selection** (330-2): hierarchical backward elimination with an R² cutoff. Reported,
  but not used to choose the model, because dropping a term moves the stationary point (D-18).
* **Requirements** (330-3/4): at least as many observations as terms plus one (rule of thumb:
  five per term); a factor raised to power k needs k + 1 distinct values; and the
  multiple-regression assumptions, *including independent errors*. Hourly data violate that
  last one (D-15).
* **Coding** (330-4): NCSS does not code factors; its example was hand-coded to −1/0/+1. Here
  the 1st and 99th percentiles map to ±1, which changes coordinates but not the fitted
  function (D-13).
* **Reports** (330-5 to 330-14): descriptive statistics; the hierarchical model summary; the
  sequential ANOVA (linear, then quadratic, then cross products) with incremental R²; the same
  table against pure error; ANOVA by factor; estimates with last R²; the optimum, found by
  numerical search and not constrained to the design space; residuals; contour plots.
* **Lack of fit** (330-1, 330-8): pure error comes from exact duplicates, which hourly data
  essentially never have, so near-replicate cells are used instead, posed as a nested test
  (D-16).

**Where the textbook procedure does not fit hourly ERCOT data**, and how each is handled:
1. Errors are serially dependent: Newey–West (HAC) inference and out-of-sample validation
   (D-15, D-25, D-26).
2. Exact duplicates do not exist: near-replicate lack of fit (D-16).
3. The factors are correlated, so the design region is a curved band rather than a cube:
   surfaces are read only where data exist, and individual coefficients are not interpreted
   (D-11, D-33).
4. Heavy tails dominate least squares: the raw spread is kept as the response, with
   winsorised and transformed fits as sensitivities (D-14).

### Other reading

The literature that motivates the study, and how the results compare with it, is in
`LITERATURE.md`. The report cites the method references (NCSS; Myers, Montgomery and
Anderson-Cook 2016; Newey and West 1987) and two results it examines directly: Engle et al.
(1986), on the nonlinear weather–demand relationship, and Bessembinder and Lemmon (2002), on
why the size of the spread matters as well as its sign.

## 9 October 2026: the 2025 data

Pulled, validated and assembled the full year (details in `DATA.md`; audit in
`data/processed/<window>/audit/`).

* **Panel.** 35,040 zone-hours (8,760 per zone), joined on UTC timestamps so the 23- and
  25-hour daylight-saving days are kept. No zone-hour is missing or duplicated, and every one
  is usable for estimation (D-21).
* **Requests.** All four zones in one filtered request per month rather than Houston first, at
  the same cost (D-20).
* **Cross-checks (D-23).**
  * Henry Hub gas from EIA matches FRED on every date.
  * Station temperatures agree with NOAA records to within 0.08 °F (Houston Intercontinental
    in July, Dallas–Fort Worth in January).
  * The spread's sign and values match Grid Status's own DA–RT series exactly on a test week
    (D-19).
  * Forecast-zone load correlates at 0.997 with ERCOT's native load, but runs about 700 MW
    higher. The gap is largest overnight, grows through the autumn and nearly vanishes after
    RTC+B, which fits battery charging being counted in telemetered load; that explanation is
    plausible but unverified (D-22). Forecast-zone load also includes the municipal and
    cooperative utilities that settle at their own load zones, so it covers a larger footprint
    than the matching price (D-03).
* **Proxies.** Henry Hub is a Louisiana benchmark rather than a delivered Texas fuel cost
  (D-08), and two to six airport stations per zone are coarse stand-ins for a mesoscale
  temperature field (D-06).

## 10 October 2026: exploratory analysis

(`EDA_FINDINGS.md`, Q1–Q5.)

* **The spread.** Small DA premium in the typical hour (median +$1.9 to +$2.8; DA above RT in
  59–63% of hours), means near zero, and extremely heavy, asymmetric tails: skewness −18 to
  −25, and the 1% largest hours carry 17–21% of all |spread|. Because those tails are the
  object of interest, the primary response stays unwinsorised (D-14, revised).
* **Load and temperature.** V-shaped in Houston, North and South, with the minimum near
  57–61 °F; T + T² explains 85–89% of load there, against 12–46% for a straight line. West's
  load is largely industrial (R² 0.32). The second-order terms are directly motivated, and so
  is the collinearity problem: within each branch of the V, temperature and load correlate at
  0.70–0.94 in absolute value in the eastern zones.
* **Dispersion.** U-shaped in temperature: no zone has a single top-1% |spread| hour between
  55 and 65 °F. The cold-end size rests largely on the 19–20 February storm; the U-shape
  survives its exclusion, its magnitude does not.
* **Zones.** Extreme hours are mostly system-wide (70 of 156 hours with any extreme have three
  or four zones extreme at once), so the four zone fits are not independent evidence. West's
  ordinary hours behave differently from the other zones.
* **Timing.** 75% of extreme hours fall in the morning and evening ramps, and all of them fall
  on 56 of 365 days. Hour of day therefore has to be controlled for, so the weather surface is
  not credited with ramp structure.

**Factor decision (D-11).** Temperature, actual zonal load and Henry Hub gas in a full
quadratic. The assignment is about temperature, demand and price, so actual load stays in
despite its collinearity with temperature. The collinearity is handled and reported (coding,
variance inflation factors, reading surfaces only where data exist, no coefficient-by-
coefficient interpretation), not avoided. A load-anomaly specification is fitted as a
comparison and changes no conclusion.

## 10 October 2026: the response-surface ladder

(`MODEL_RESULTS.md`, sections 1–4.)

**Models.** M0 is the hour-of-day mean (the benchmark); M1 is linear in T, L and G; M2 adds
the three squares and three interactions (the RSM); M3 adds hour-of-day indicators to M2.
Three responses per zone: the signed spread, its magnitude |spread|, and the DA implied heat
rate.

**Inference and validation.**
* Effective sample sizes for estimating a mean are only 70–79 for temperature and 102–124 for
  load out of 8,760 hours, so classical standard errors badly overstate precision. Inference
  uses HAC with 48 lags and joint Wald tests on groups of terms (D-26).
* Forward expanding-window validation is primary: train on all months before the test month,
  test April–December (6,601 hours per zone). Leave-one-month-out is secondary (D-25). Both use
  realised conditions, so both measure stability of an explanation, not forecasting skill.

**Results.**
* **Signed spread.** Barely explained (R² at most 5%), and the quadratic terms make later-month
  error worse. A negative result.
* **Spread magnitude.** Curvature is significant in every zone under HAC inference, and the
  5th–95th percentile band of the spread widens markedly from mild to the hottest hours. But no
  model beats the hourly benchmark out of sample: extremes are too few and episodic.
* **DA heat rate.** Load and time of day explain most of it.

## 10 October 2026: forecast surprises

The DA market clears on expectations, so the spread should respond to the *surprise*, not the
realised level. Pre-specified before any surprise model was fitted (D-28 to D-30):

* **Information set.** The latest forecasts published by 09:45 on the day before, which
  precedes the 10:00 DA bid close on every day: the 09:30 load forecast and the 08:55 wind and
  solar forecasts. All 26,280 selected rows were published before the cutoff.
* **Renewable surprises against available capacity, not generation.** Generation-based errors
  are biased by economic curtailment at negative prices, which is caused by price and would
  contaminate the regressor.
* **A ladder with an evidence rule.** An added group counts only if its Holm-adjusted HAC test
  is significant *and* forward validation improves. In-sample ΔR² never counts.

**Results.** The net-load surprise has a rank correlation of −0.35 to −0.47 with the spread,
against −0.01 to +0.09 for realised temperature and load. A two-factor surprise surface S (the
surprise and expected net load) is the only model with positive forward skill against the
hourly benchmark (Houston +0.054, North +0.038, West +0.045; South −0.039). Flexible loads
curtail at high RT prices, so the load surprise is partly endogenous: these are associations.

## 10 October 2026: settled prices

Switched the price source to ERCOT's final settled archives (NP4-180-ER, NP6-785-ER), with Grid
Status's first-published series as the cross-check (D-31). Every one of the 35,040 DA hours and
140,160 RT intervals matches by key, but values differ for 89 DA hours on two dates and 135 RT
intervals on eight dates, all days ERCOT corrected afterwards. The largest is an RT interval on
30 June that first appeared at $1,624–$1,636/MWh but settled at $28–$40: a phantom spike that
would otherwise have entered the extreme-spread analysis. Re-running everything on settled
prices moved no conclusion.

## 10 October 2026: the 2024 replication and the West basis

**2024 through the identical pipeline** (`MODEL_RESULTS.md`, replication sections). Prices,
load, weather and gas are complete for all 8,784 hours. A −63 °F reading at Brownsville on 3
April 2024 was a sensor fault, which led to a physical-plausibility filter (−20 to 125 °F)
before station averaging.

* **Replicates:** the DA premium in most hours, negative skew, the V-shaped load, the
  curvature of spread magnitude, the U-shape of dispersion in temperature, and the
  concentration of extremes in ramp hours and, later, in tight hours.
* **Does not replicate:** the hottest-hour DA premium in South and West, the heat-rate model's
  out-of-sample skill (+0.09 to +0.13 in 2025, −0.19 to +0.01 in 2024), and the strength of the
  surprise-by-tightness interaction.
* **Coverage limit, found later the same day:** ERCOT's available-capacity forecasts are
  complete only from 24 September 2024, so the surprise results could not be replicated, and
  the fit-2024/predict-2025 test trains on 2,364 hours rather than a full year (D-36).

**West basis (D-32).** The spread splits exactly into a hub-average term and a basis term (the
DA-to-RT change in the zone's price relative to the hub average). West's typical hour carries a
basis surprise three to five times the other zones', concentrated at low West load and high
DA-time wind forecasts, consistent with export-constrained, high-wind conditions. West's
extremes, however, remain mostly system-wide. Descriptive only; constraint-level data would be
needed to test the mechanism.

## 10 October 2026: matched comparisons and the report's figures

Drafting the report raised two questions that changed what it says:

* **What does curvature add, holding everything else fixed? (D-34)** M3 had been compared only
  with the hourly benchmark and with M2. Adding M1H (linear, with the same hour block) isolates
  the six second-order terms. For the DA heat rate in Houston, North and South, M1H forecasts
  *better* than M3 (skill +0.12 to +0.17, against +0.09 to +0.13), so the earlier attribution
  of the gain to the response surface was wrong for those zones. Only West gains from curvature.
  The report's Table 3 is built on these matched pairs.
* **Where can a surface be read? (D-33)** The convex hull of a V-shaped data cloud includes the
  empty inside of the V. Surfaces are therefore drawn only where at least five comparable hours
  lie nearby; extrapolation is grey, and fitted magnitudes below zero are hatched, because an
  unconstrained quadratic for a non-negative outcome can predict impossible values (6.3% of
  West's fitted hours).
* **Load curvature out of sample (D-36).** In the same month-by-month test, the squared
  temperature term cuts later-month load error by 84–86% in Houston, North and South, and by
  25% in West.

## 10 October 2026: extreme hours and the 16 May event

(`EDA_FINDINGS.md`, Q6; D-35.)

**Definitions.** An hour is extreme when its |spread| is in the zone's own top 1%, so the base
rate is 1% by construction and comparisons are relative risks. Tightness is system net load
(load minus wind and solar generation). Intervals come from a day-block bootstrap, because
extremes cluster on few days.

* **Headline.** Extremes occur in 6.0% of the tightest 10% of hours against 0.45% of the rest:
  about 13 times as often (90% interval 8–21), or 11 times without the February storm. 2024
  gives 12 (7–19).
* **Time of day.** Tight hours are mostly summer evenings, so the comparison was repeated with
  net load ranked within each hour of day. The ratio roughly halves, to 7.4 (4.2–12.5): not
  explained by time of day alone. Season, weather, renewables and outages are not held fixed,
  so this is a robustness check and a descriptive association, not a causal effect.
* **In dollars.** Because the 13× is a relative risk, the same comparison was made at fixed
  thresholds: |spread| above $100/MWh in 5.3% of tight hours against 0.39% of the others.

**The event day.** Chosen retrospectively by a fixed rule: the day with the most RT-above-DA
extreme hours in the tightest decile, outside the February storm. That is 16 May 2025. The
system ran tighter than the forecast the DA market cleared on from the early morning onward;
Houston's RT price reached $1,507/MWh in the hour beginning 19:00 against a DA price of $184.
The day also shows the limits of the explanation: the largest surprise (6.7 GW, at 09:00) came
with an ordinary spread, and the surprise surface, fitted on January–April, predicted −$1.90
for the 19:00 hour against an observed −$1,324: the right sign, nowhere near the size.

## 10 October 2026: calendar controls and net-load ramps (isolated experiment)

(`experiments/RSM_CONTROLS.md`.) A timeboxed test of whether calendar dummies, seasonal Fourier
terms or net-load ramps should be added to the RSM, run without touching the report, on the
same observations and folds, with training-only preprocessing.

**Result: no addition passes all three criteria (in-year forward gain, stability across years,
interpretability).** Calendar controls help the DA heat rate within 2025 but not when fitting
on 2024 and predicting 2025. Ramps have the expected signs but half their variance is time of
day, and their small gain reverses out of year. Seasonal terms are harmful with only one year of
expanding-window training. The report's specification was left unchanged.

## 10 October 2026: the final report

Restructured the report around the assignment's steps (data, required measures, exploratory
analysis, response surface regression, results, conclusions) and revised it through several
accuracy passes. Corrections made along the way, each checked against the CSVs:

* A linear model in T, L and G *can* predict larger magnitudes at both temperature extremes,
  through the V-shaped load term; the report's comparison of linear and quadratic fits was
  rewritten accordingly.
* The Grid Status reconciliation: two corrected DA dates and eight corrected RT dates.
* The 16 May chronology: the system ran tighter than expected from the hour beginning 03:00;
  Houston had extreme hours from 16:00, and all four zones did from 18:00 to 20:00.
* The 2024 replication is described as partial, because of the forecast coverage limit.
* Throughout: associations, not causes; explanation with realised conditions, not forecasting.

The final report is ten pages: `reports/report_v2.pdf`.

## Open questions and next steps

* **Mesoscale temperature.** Replace station averages with a gridded field averaged over each
  zone, as the project title suggests (D-06, D-07).
* **Weather-forecast errors.** The surprise results suggest the spread responds to what was
  unexpected; temperature forecast errors are the natural weather counterpart.
* **Regional gas.** Houston Ship Channel and Waha prices instead of Henry Hub (D-08).
* **Spikes.** Reserve-margin and outage data would be the targeted additions for explaining the
  extremes; more polynomial terms alone have not helped.
* **More years.** Seasonal terms and cross-year stability need several years of data, with
  complete forecast archives.
* **West.** Constraint-level data (shadow prices) to test the export-congestion explanation.